# Hinglish reliability runs
Settings (right sidebar): **Accelerator = GPU T4 x2 or P100**, **Internet = On**. Then Run All (or *Save Version -> Save & Run All* to run in the background).

In [ ]:
import glob, os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=False)
SCRIPT = glob.glob("/kaggle/input/**/train_predict.py", recursive=True)[0]
DATA = os.path.dirname(SCRIPT)
OUT = "/kaggle/working/out"
print("script:", SCRIPT, "| data dir:", DATA, "| files:", sorted(os.listdir(DATA)))
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable accelerator!")

In [ ]:
def run(model, seed, extra=""):
    tag = f"{model.split('/')[-1]}_s{seed}" + ("_aug" if "--augment" in extra else "")
    if os.path.exists(f"{OUT}/{tag}/preds.npz"):
        print("skip (done):", tag); return
    cmd = f"python {SCRIPT} --model {model} --seed {seed} --data {DATA} --out {OUT} {extra}"
    print(">>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    print("exit code", r.returncode, flush=True)

## 1. Smoke test (~1-2 min). Must finish without errors before continuing.

In [ ]:
run("xlm-roberta-base", 99, "--limit 200 --epochs 1")

## 2. Priority run: XLM-R seed 0 (~15-25 min). Look at dev/test accuracy: expect roughly 0.60-0.70 on SentiMix.

In [ ]:
run("xlm-roberta-base", 0)

## 3. IndicBERT seed 0 and the variant-augmented baseline

In [ ]:
run("ai4bharat/indic-bert", 0)
run("xlm-roberta-base", 0, "--augment")

## 4. Extra seeds (for confidence intervals and the deep-ensemble baseline)

In [ ]:
for s in (1, 2):
    run("xlm-roberta-base", s)
    run("ai4bharat/indic-bert", s)

## 5. Package results (download `out.zip` from the Output tab)

In [ ]:
import shutil, glob
for d in glob.glob(f"{OUT}/*_s99"): shutil.rmtree(d, ignore_errors=True)   # drop smoke-test run
shutil.make_archive("/kaggle/working/out", "zip", OUT)
print(sorted(os.listdir(OUT)))